# Segment Beetles -- Single-Image Test Bench

A quick way to test the watershed background-removal algorithm on **one photo at a time** and see exactly how well it's doing, before ever running it across a whole dataset. Every intermediate stage of the algorithm is shown side by side, so you can see *where* it's succeeding or failing (e.g. missing legs, catching a ruler, etc.) rather than just the final result.

**How to use this notebook:**
1. Run **Setup**, then **Step 1** to pick an image file (a picker window will pop up -- it may open behind your other windows).
2. Run **Step 2** once to define the segmentation function.
3. Run **Step 3** to segment the current image and see the result. Tweak the parameters at the top of that cell and re-run it as many times as you like -- no need to re-pick the image each time.
4. Re-run **Step 1** whenever you want to switch to a different photo.

## Setup
Requires OpenCV (`opencv-python`), NumPy, and Matplotlib, in addition to the packages already used elsewhere in this project.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tkinter as tk
from tkinter import filedialog

## Step 1 -- choose an image
Pick a single photo to test against. Re-run this cell any time you want to switch images.

In [ ]:
root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)

IMAGE_PATH = filedialog.askopenfilename(
    title="Choose an image to test segmentation on",
    filetypes=[("Images", "*.jpg *.jpeg *.png *.bmp *.tif *.tiff"), ("All files", "*.*")],
)
root.destroy()

if not IMAGE_PATH:
    raise SystemExit("No image selected -- re-run this cell and choose a file.")

print(f"Selected: {IMAGE_PATH}")

## Step 2 -- the segmentation function

Marker-based watershed segmentation:
1. Otsu-threshold the grayscale image (inverted, since we assume a dark specimen on a light background).
2. Clean up noise with morphological opening -- but only to seed a conservative "definitely foreground" core; thin parts like antennae get eroded away here on purpose.
3. Get a "sure background" region by dilating the **raw threshold**, not the opened mask -- if we dilated the opened mask instead, anything opening erased (e.g. antennae) would be permanently locked in as background before watershed even runs, since watershed can only adjudicate the ambiguous "unknown" ring between sure-fg and sure-bg. Using the raw threshold keeps thin appendages inside that ambiguous ring instead, so watershed's own gradient-based flooding gets a chance to correctly claim them.
4. Get a "sure foreground" region from the distance transform -- the pixels deepest inside the specimen.
5. Whatever's left between the two is the "unknown" boundary region for watershed to resolve.
6. Run `cv2.watershed`, then keep every resulting foreground region big enough to not be noise (occasionally a specimen's legs/antennae can still end up as a separate label from its body) and paint everything else white.
7. **Strip printed rule lines** (specimen labels, rulers, grid backgrounds) that survived into the mask -- even ones a leg or antenna crosses directly. This uses the standard Hough transform to find long, strongly-supported straight lines anywhere in the image (a real specimen edge essentially never racks up enough votes at a single angle+offset the way a long printed line does). For each detected line, we measure its typical cross-sectional thickness at many points along its length -- since most of a line's length isn't covered by the specimen, this "baseline" thickness is reliable -- and only erase points that still match that baseline. A leg or antenna crossing the line is locally much wider than the line's own baseline thickness, so it's left alone even exactly where it crosses.

This version returns every intermediate stage (not just the final result) so Step 3 can display them all for inspection.

**Known limitation:** the line-stripping step judges each point along a detected line independently, so on rare occasions a single point where an appendage's width briefly and coincidentally matches the line's own thickness can still get clipped (e.g. a thin tapering leg segment). If you see this, it's a real limitation to watch for -- not a bug to work around with parameters.

In [ ]:
def _perpendicular_run_length(binary_img, x, y, dx, dy, max_r=80):
    """Contiguous run of foreground pixels centered at (x, y) along direction (dx, dy)."""
    h, w = binary_img.shape
    def at(px, py):
        if 0 <= px < w and 0 <= py < h:
            return binary_img[py, px] > 0
        return False
    if not at(int(round(x)), int(round(y))):
        return 0
    length = 1
    for sign in (1, -1):
        r = 1
        while r <= max_r:
            px, py = x + sign * dx * r, y + sign * dy * r
            if not at(int(round(px)), int(round(py))):
                break
            length += 1
            r += 1
    return length


def _strip_rule_lines(mask, thresh, protected_core, w, h, line_vote_ratio=0.35, width_outlier_ratio=1.8):
    """Detects long, straight, strongly-supported lines (printed labels, rulers, grid backgrounds)
    and erases only the parts of `mask` that still match that line's own typical thickness --
    a leg/antenna crossing the line is locally much wider than the line itself, so it survives."""
    edges = cv2.Canny(thresh, 20, 60)
    vote_threshold = int(line_vote_ratio * max(w, h))
    hough_lines = cv2.HoughLines(edges, 1, np.pi / 180, threshold=vote_threshold)

    suppress = np.zeros_like(mask)
    if hough_lines is None:
        return suppress, 0

    diag = int(np.hypot(w, h))
    n_lines_used = 0
    for rho_theta in hough_lines:
        rho, theta = rho_theta[0]
        a, b = np.cos(theta), np.sin(theta)
        x0, y0 = a * rho, b * rho
        dx_line, dy_line = -b, a
        dx_perp, dy_perp = a, b

        ts = np.arange(-diag, diag, 2)
        xs = x0 + dx_line * ts
        ys = y0 + dy_line * ts
        valid = (xs >= 0) & (xs < w) & (ys >= 0) & (ys < h)
        xs, ys = xs[valid], ys[valid]
        if len(xs) < 10:
            continue

        run_lengths = np.array([_perpendicular_run_length(thresh, x, y, dx_perp, dy_perp) for x, y in zip(xs, ys)])
        on_line = run_lengths > 0
        if on_line.sum() < 10:
            continue
        baseline = np.median(run_lengths[on_line])
        n_lines_used += 1

        for x, y, rl in zip(xs, ys, run_lengths):
            if rl == 0 or rl > baseline * width_outlier_ratio:
                continue  # not on this line, or a much wider crossing structure (leg) -- protect it
            r = rl / 2
            p1 = (int(round(x - dx_perp * r)), int(round(y - dy_perp * r)))
            p2 = (int(round(x + dx_perp * r)), int(round(y + dy_perp * r)))
            cv2.line(suppress, p1, p2, 255, thickness=2)

    suppress = cv2.bitwise_and(suppress, cv2.bitwise_not(protected_core))
    return suppress, n_lines_used


def segment_insect(image_path, open_kernel_size=3, open_iterations=2, dilate_iterations=3,
                    dist_ratio=0.4, noise_ratio=0.02, min_fg_ratio=0.001,
                    line_vote_ratio=0.35, width_outlier_ratio=1.8):
    """Returns (stages, succeeded, reason). `stages` is a dict of every intermediate image,
    always including 'original' and 'result' even on failure (in which case 'result' == original)."""
    img = cv2.imread(str(image_path))
    if img is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")

    stages = {"original": img}
    h, w = img.shape[:2]

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    stages["threshold"] = thresh

    kernel = np.ones((open_kernel_size, open_kernel_size), np.uint8)
    opening = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=open_iterations)
    stages["opening"] = opening

    # Dilate the RAW threshold (not the opened mask) so thin parts erased by opening -- e.g.
    # antennae -- stay inside the ambiguous "unknown" ring below, instead of being locked in
    # as sure background before watershed ever gets a chance to correctly claim them.
    sure_bg = cv2.dilate(thresh, kernel, iterations=dilate_iterations)
    stages["sure_bg"] = sure_bg

    dist_transform = cv2.distanceTransform(opening, cv2.DIST_L2, 5)
    stages["distance_transform"] = dist_transform
    if dist_transform.max() == 0:
        stages["result"] = img
        return stages, False, "no foreground detected by thresholding"

    _, sure_fg = cv2.threshold(dist_transform, dist_ratio * dist_transform.max(), 255, 0)
    sure_fg = np.uint8(sure_fg)
    stages["sure_fg"] = sure_fg

    unknown = cv2.subtract(sure_bg, sure_fg)

    n_labels, markers = cv2.connectedComponents(sure_fg)
    if n_labels <= 1:
        stages["result"] = img
        return stages, False, "no distinct foreground regions found"
    markers = markers + 1
    markers[unknown == 255] = 0

    markers = cv2.watershed(img, markers)
    boundary_overlay = img.copy()
    boundary_overlay[markers == -1] = (0, 0, 255)
    stages["boundary_overlay"] = boundary_overlay

    # Background is label 1, watershed boundaries are -1, each specimen/blob is its own label > 1.
    # A single specimen can occasionally end up split across several labels here, so we keep
    # every label that isn't tiny noise (dust specks, stray marks) rather than only the single
    # largest one -- otherwise real parts of the specimen get painted over as background.
    fg_labels, counts = np.unique(markers[markers > 1], return_counts=True)
    if len(fg_labels) == 0:
        stages["result"] = img
        return stages, False, "watershed found no foreground region"

    total_fg_area = counts.sum()
    if total_fg_area < min_fg_ratio * img.shape[0] * img.shape[1]:
        stages["result"] = img
        return stages, False, "detected specimen region too small -- likely a failed segmentation"

    noise_cutoff = max(noise_ratio * counts.max(), 15)
    keep_labels = fg_labels[counts >= noise_cutoff]

    # Include the watershed boundary line itself so we don't leave a thin white outline around the specimen.
    mask = np.uint8(np.isin(markers, keep_labels) | (markers == -1)) * 255
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    stages["mask_before_line_removal"] = mask

    # Strip printed rule lines (labels, rulers, grid backgrounds) that survived into the mask,
    # even where a leg/antenna crosses directly over them. See Step 2's markdown for how.
    protected_core = cv2.dilate(sure_fg, np.ones((15, 15), np.uint8), iterations=2)
    line_suppress, n_lines_used = _strip_rule_lines(
        mask, thresh, protected_core, w, h,
        line_vote_ratio=line_vote_ratio, width_outlier_ratio=width_outlier_ratio,
    )
    stages["line_suppress"] = line_suppress
    mask = cv2.bitwise_and(mask, cv2.bitwise_not(line_suppress))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=1)
    stages["mask"] = mask

    result = img.copy()
    result[mask == 0] = (255, 255, 255)
    stages["result"] = result
    return stages, True, ""

## Step 3 -- run it and inspect every stage

Edit the parameters below and re-run this cell to see the effect immediately -- no need to re-pick the image.

- `OPEN_KERNEL_SIZE` / `OPEN_ITERATIONS`: how aggressively small noise (and thin specimen parts, like legs) gets stripped out before the distance transform.
- `DILATE_ITERATIONS`: how far the "definitely background" region grows inward -- higher shrinks the specimen's safe zone.
- `DIST_RATIO`: how deep into the specimen a pixel must be to count as "definitely foreground" (0.4 = the innermost 60% by distance). Lower this to keep more of the specimen's thinner parts (legs/antennae).
- `NOISE_RATIO`: how small a detected blob can be, relative to the largest one, before it's discarded as noise rather than kept as part of the specimen.
- `MIN_FG_RATIO`: the smallest fraction of the whole image a detected specimen is allowed to take up before the result is flagged as a likely failure.
- `LINE_VOTE_RATIO`: how strongly-supported a straight line must be (as a fraction of the image's diagonal-ish size) before it's treated as a printed rule line rather than incidental specimen edges. Lower this if a faint ruler/label line isn't being detected at all; raise it if real specimen edges are being mistaken for rule lines.
- `WIDTH_OUTLIER_RATIO`: how many times wider than a line's own typical thickness a crossing structure must be before it's protected as "probably a leg/antenna, not the line." Lower this if legs are still getting clipped where they cross a line; raise it if the line isn't being fully removed.

In [ ]:
OPEN_KERNEL_SIZE = 3
OPEN_ITERATIONS = 2
DILATE_ITERATIONS = 3
DIST_RATIO = 0.4
NOISE_RATIO = 0.02
MIN_FG_RATIO = 0.001
LINE_VOTE_RATIO = 0.35
WIDTH_OUTLIER_RATIO = 1.8

stages, ok, reason = segment_insect(
    IMAGE_PATH,
    open_kernel_size=OPEN_KERNEL_SIZE,
    open_iterations=OPEN_ITERATIONS,
    dilate_iterations=DILATE_ITERATIONS,
    dist_ratio=DIST_RATIO,
    noise_ratio=NOISE_RATIO,
    min_fg_ratio=MIN_FG_RATIO,
    line_vote_ratio=LINE_VOTE_RATIO,
    width_outlier_ratio=WIDTH_OUTLIER_RATIO,
)

print("Segmentation succeeded" if ok else f"Segmentation FLAGGED: {reason}")

def to_rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

panels = [
    ("Original", to_rgb(stages["original"]), None),
    ("Otsu threshold", stages.get("threshold"), "gray"),
    ("After opening (noise removed)", stages.get("opening"), "gray"),
    ("Sure background", stages.get("sure_bg"), "gray"),
    ("Distance transform", stages.get("distance_transform"), "viridis"),
    ("Sure foreground", stages.get("sure_fg"), "gray"),
    ("Watershed boundaries (red)", to_rgb(stages["boundary_overlay"]) if "boundary_overlay" in stages else None, None),
    ("Mask before line removal", stages.get("mask_before_line_removal"), "gray"),
    ("Detected rule-line pixels removed", stages.get("line_suppress"), "gray"),
    ("Final mask", stages.get("mask"), "gray"),
    ("Result", to_rgb(stages["result"]), None),
]

fig, axes = plt.subplots(3, 4, figsize=(17, 13))
for ax, (title, img, cmap) in zip(axes.flat, panels):
    if img is None:
        ax.set_visible(False)
        continue
    ax.imshow(img, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.axis("off")
for ax in axes.flat[len(panels):]:
    ax.set_visible(False)
fig.tight_layout()
plt.show()

## Step 4 -- save the result

Saves the full stage-by-stage grid shown above (the same figure, original through result) as a single PNG into `Data/segmented_test/`, created automatically next to this repo's other `Data/` folders. The filename is derived from the source image, so re-running this on a different image just adds another file rather than overwriting -- only saving over the *same* source image twice will overwrite its previous grid.

In [ ]:
def find_repo_root(marker='Models'):
    """Walk upward from the current working directory until a folder containing
    `marker` (i.e. the repo's Models/ folder) is found, so paths below don't
    depend on whose machine this notebook is running on."""
    path = os.path.abspath(os.getcwd())
    for _ in range(5):
        if os.path.isdir(os.path.join(path, marker)):
            return path
        parent = os.path.dirname(path)
        if parent == path:
            return os.path.abspath(os.getcwd())
        path = parent
    return os.path.abspath(os.getcwd())

REPO_ROOT = find_repo_root()
OUTPUT_DIR = os.path.join(REPO_ROOT, "Data", "segmented_test")
os.makedirs(OUTPUT_DIR, exist_ok=True)

base_name, _ = os.path.splitext(os.path.basename(IMAGE_PATH))
out_path = os.path.join(OUTPUT_DIR, f"{base_name}_stages.png")
fig.savefig(out_path, dpi=150)
print(f"Saved to: {out_path}")